In [1]:
import os 


In [3]:
os.chdir('/Users/praptidinesh/Documents/Monash/FIT5120 IE/ITERATION/DATA')

In [5]:
os.getcwd()

'/Users/praptidinesh/Documents/Monash/FIT5120 IE/ITERATION/DATA'

In [ ]:
import pandas as pd

# Step 1: Load the CSV
video_games_path = 'video_games_1 copy.csv'  # <-- Update your local path here
video_games_df = pd.read_csv(video_games_path)

# Step 2: Drop unnecessary columns
video_games_cleaned = video_games_df.drop(columns=[
    'index', 'Rank', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales'
])

# Step 3: Rename columns to make them database-friendly
video_games_cleaned = video_games_cleaned.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year': 'year_of_release',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Global_Sales': 'global_sales'
})

# Step 4: Drop rows with missing important fields
video_games_cleaned = video_games_cleaned.dropna(subset=[
    'name', 'platform', 'year_of_release', 'genre'
])

# Step 5: Standardize the genre text
video_games_cleaned['genre'] = video_games_cleaned['genre'].str.strip().str.title()

# (Optional) Step 6: Save cleaned data to new CSV
video_games_cleaned.to_csv('cleaned_video_games.csv', index=False)

# Quick check
print(video_games_cleaned.head())


In [ ]:
import pandas as pd

# Step 1: Load the dataset
online_behavior_path = 'online_gaming_behavior_dataset.csv'
online_behavior_df = pd.read_csv(online_behavior_path)

# Step 2: Drop unnecessary columns
online_behavior_cleaned = online_behavior_df.drop(columns=[
    'Location', 
    'InGamePurchases', 
    'GameDifficulty', 
    'AvgSessionDurationMinutes', 
    'PlayerLevel', 
    'AchievementsUnlocked'
])

# Step 3: Rename columns for SQL-friendly names
online_behavior_cleaned = online_behavior_cleaned.rename(columns={
    'PlayerID': 'player_id',
    'Age': 'age',
    'Gender': 'gender',
    'GameGenre': 'game_genre',
    'PlayTimeHours': 'playtime_hours',
    'SessionsPerWeek': 'sessions_per_week',
    'EngagementLevel': 'engagement_level'
})

# ⭐ ⭐ ⭐ Add genre mapping fix here ⭐ ⭐ ⭐
genre_mapping = {
    'RPG': 'Role-Playing',
    'Strategy': 'Strategy',
    'Simulation': 'Simulation'  # New genre, handled separately
}

online_behavior_cleaned['game_genre'] = online_behavior_cleaned['game_genre'].replace(genre_mapping)

# Step 4: Normalize engagement_level
online_behavior_cleaned['engagement_level'] = online_behavior_cleaned['engagement_level'].str.strip().str.capitalize()

# Step 5: Ensure correct data types
online_behavior_cleaned['age'] = online_behavior_cleaned['age'].astype(int)
online_behavior_cleaned['playtime_hours'] = online_behavior_cleaned['playtime_hours'].astype(float)
online_behavior_cleaned['sessions_per_week'] = online_behavior_cleaned['sessions_per_week'].astype(int)

# (Optional) Step 6: Save cleaned dataset
online_behavior_cleaned.to_csv('cleaned_online_gaming_behavior.csv', index=False)

# Step 7: Quick check
print(online_behavior_cleaned['game_genre'].unique())


In [ ]:
import pandas as pd

# Load your cleaned online gaming behavior data
behavior_data_path = 'cleaned_online_gaming_behavior.csv'  # <-- Update this path
behavior_df = pd.read_csv(behavior_data_path)

# Extract only the needed columns for users table
users_df = behavior_df[['player_id', 'age', 'gender']]  # <-- lowercase!

# Rename columns to match users table
users_df = users_df.rename(columns={
    'player_id': 'user_id',
    'age': 'age',
    'gender': 'gender'
})

# Save as a new CSV
users_df.to_csv('cleaned_users.csv', index=False)

# Quick check
print(users_df.head())


In [ ]:
import pandas as pd

# Load your cleaned online gaming behavior data
behavior_data_path = 'cleaned_online_gaming_behavior.csv'  # <-- Update this path
behavior_df = pd.read_csv(behavior_data_path)

# Extract only needed columns for sessions table
sessions_df = behavior_df[['player_id', 'game_genre', 'playtime_hours', 'sessions_per_week', 'engagement_level']]

# Rename player_id to user_id
sessions_df = sessions_df.rename(columns={
    'player_id': 'user_id'
})

# Save as a new CSV
sessions_df.to_csv('cleaned_sessions.csv', index=False)

# Quick check
print(sessions_df.head())


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("steam.csv")

# Convert release_date to datetime
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Drop rows with invalid dates (if any)
df = df.dropna(subset=['release_date'])

# Convert delimited string fields into lists
for col in ['categories', 'genres', 'steamspy_tags']:
    df[col] = df[col].fillna('').apply(lambda x: x.split(';') if isinstance(x, str) else [])

# Extract lower bound of owners (convert to integer)
df['owners_lower'] = df['owners'].str.extract(r'(\d+)').astype(float)

# Remove unneeded columns
df_cleaned = df.drop(columns=['appid', 'developer', 'publisher', 'english'])

# Optional: create a flag for potentially violent genres
violent_keywords = [
    "Action", "Shooter", "FPS", "Combat", "Violent", "War", 
    "Gore", "Military", "Battle", "Blood", "Fighting", "PvP"
]

# Convert all to lowercase for consistency
violent_keywords = [k.lower() for k in violent_keywords]

df_cleaned['is_potentially_violent'] = df_cleaned['genres'].apply(
    lambda genres: any(keyword in genre.lower() for keyword in violent_keywords for genre in genres)
)


# Save cleaned version (optional)
df_cleaned.to_csv("cleaned_steam_data.csv", index=False)


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("steam.csv")

# Rename 'name' to 'game_title'
df.rename(columns={'name': 'game_title'}, inplace=True)

# Step 1: Basic Cleaning
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date'])  # Remove invalid dates

# Step 2: Split multi-value string columns into lists
for col in ['categories', 'genres', 'steamspy_tags']:
    df[col] = df[col].fillna('').apply(lambda x: x.split(';') if isinstance(x, str) else [])

# Step 3: Extract lower bound of owners range
df['owners_lower'] = df['owners'].str.extract(r'(\d+)').astype(float)

# Step 4: Define violence-related keywords
violent_keywords_strong = ['FPS', 'Blood', 'War', 'Violent', 'Gore']
violent_keywords_mild = ['Shooter', 'Combat', 'Fighting', 'Battle', 'PvP']

# Step 5: Classification function for violence level
def classify_violence(row):
    tags = row['genres'] + row['steamspy_tags']
    categories = row['categories']
    required_age = row['required_age']
    score = 0

    # Apply keyword scoring
    score += sum(tag.lower() in [k.lower() for k in violent_keywords_strong] for tag in tags) * 2
    score += sum(tag.lower() in [k.lower() for k in violent_keywords_mild] for tag in tags)

    # Bonus points for multiplayer combat setups
    if 'multiplayer' in [cat.lower() for cat in categories] and 'shooter' in [tag.lower() for tag in tags]:
        score += 2

    # Age restriction contribution
    if required_age >= 18:
        score += 2

    # Final classification
    if score >= 5:
        return "Highly Violent"
    elif score >= 3:
        return "Moderately Violent"
    else:
        return "Non-Violent"

# Step 6: Apply classification
df['violence_level'] = df.apply(classify_violence, axis=1)

# Step 7: Select and format final DataFrame
df_cleaned = df[['game_title', 'release_date', 'platforms', 'required_age', 'categories',
                 'genres', 'steamspy_tags', 'achievements', 'positive_ratings',
                 'negative_ratings', 'average_playtime', 'median_playtime', 'owners',
                 'price', 'owners_lower', 'violence_level']].copy()


# Step 8: Convert list columns to PostgreSQL array string format
for col in ['categories', 'genres', 'steamspy_tags']:
    df_cleaned[col] = df_cleaned[col].apply(lambda x: '{' + ','.join(tag.replace('"', '').replace("'", "") for tag in x) + '}')

# Step 9: Export to PostgreSQL-compatible CSV
df_cleaned.to_csv("classified_steam.csv", index=False)

# Optional: print first few rows
print(df_cleaned.head())


## 1. Data Loading and Cleanup 
The script begins by loading the Steam games dataset. It converts release dates into a standard date format and removes any rows where the date couldn't be properly parsed. This ensures all time-related analyses are accurate and consistent.

## 🧾 2. Parsing Multi-Value Columns
Columns like genres, categories, and user-generated tags (e.g., “FPS;Shooter;Multiplayer”) are originally stored as single strings. These are split into Python lists so we can analyze them properly—for example, checking whether a game is labeled as a "Shooter" or "PvP".

## 📊 3. Estimating Game Popularity
The owners field (which shows the estimated number of people who own a game) is originally stored as a range, like "5000000–10000000". The script extracts just the lower bound to create a numerical estimate of minimum player reach for each game.

## 🎯 4. Defining Violence Indicators
Two lists of keywords are created:

Strong indicators of violence: things like “Blood”, “Gore”, “War”

Mild indicators: words like “Combat”, “PvP”, “Shooter”

These are based on behavioral science literature linking such themes to increased aggression or stress responses, particularly in adolescents.

## 🧠 5. Scoring Each Game
Each game is assigned a violence score based on the presence of these keywords, as well as:

Whether it's labeled multiplayer and shooter (a common combo in aggressive online interactions)

Whether it's rated for adults (age 18+)

Points are given for each of these, and then a total score is calculated.

## 🏷️ 6. Classifying Violence Levels
Based on the total score, each game is classified into one of three categories:

Highly Violent: Multiple strong indicators of violence

Moderately Violent: Some signs, but not overwhelmingly violent

Non-Violent: Lacks significant aggressive or violent indicators

In [ ]:
import pandas as pd
import requests
import time

# --- STEP 1: Load Your Steam Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

# --- STEP 2: RAWG API Setup ---
API_KEY = "814164033d1945609c5c51925fdb1727"  # Replace this with your actual API key
BASE_URL = "https://api.rawg.io/api/games"

# --- STEP 3: Function to Get Violence Classification ---
def get_violence_label(title):
    try:
        params = {
            'key': API_KEY,
            'search': title,
            'page_size': 1
        }
        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown"

        game = results[0]

        # Check ESRB rating
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        # Check for tags and genres
        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]

        # Define violence heuristics
        if ('violent' in tags or 'gore' in tags or 'mature' in tags or 
            'shooter' in genres or 'action' in genres or
            esrb_name in ['mature', 'adults only']):
            return "Violent"
        else:
            return "Non-Violent"

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown"

# --- STEP 4: Apply to First 100 Games (You can remove .head(100) for full dataset) ---
violence_labels = []
for idx, title in enumerate(df['game_title'].head(100)):  # Use head(100) for testing, remove for full
    print(f"[{idx+1}] Checking: {title}")
    label = get_violence_label(title)
    violence_labels.append(label)
    time.sleep(1)  # Prevents rate limiting

df.loc[df.index[:100], 'violence_level_api'] = violence_labels

# --- STEP 5: Save to New CSV ---
df.to_csv("steam_with_api_violence_labels.csv", index=False)
print("🎉 Classification complete! File saved as 'steam_with_api_violence_labels.csv'")


In [ ]:
import pandas as pd
import requests
import time

# --- STEP 1: Load Your Steam Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

# --- STEP 2: RAWG API Setup ---
API_KEY = "814164033d1945609c5c51925fdb1727"  # Replace this with your actual API key
BASE_URL = "https://api.rawg.io/api/games"

# --- STEP 3: ACB-style Classification Function ---
def classify_acb_rating(tags, genres, esrb_rating=None):
    impact_levels = {
        'very mild': 0,
        'mild': 1,
        'moderate': 2,
        'strong': 3,
        'high': 4
    }

    # EXPANDED KEYWORDS for broader detection
    keywords = {
        'very mild': ['cartoon', 'funny', 'family-friendly'],
        'mild': ['comic mischief', 'slapstick', 'puzzle', 'fighting', 'platformer'],
        'moderate': [
            'combat', 'shooter', 'battle', 'action', 'brawler',
            'fps', 'first-person', 'arena shooter', 'third-person'
        ],
        'strong': [
            'blood', 'weapons', 'guns', 'war', 'military',
            'team-based', 'intense violence', 'class-based', 'online co-op'
        ],
        'high': [
            'gore', 'torture', 'dismemberment', 'execution',
            'mature', 'graphic violence', 'realistic violence'
        ]
    }

    tag_set = set(tags + genres)
    tag_set = {t.lower() for t in tag_set}

    highest_impact = 0
    for level, kws in keywords.items():
        for kw in kws:
            if kw in tag_set:
                highest_impact = max(highest_impact, impact_levels[level])

    # If ESRB says Mature or Adults Only, enforce max rating
    if esrb_rating:
        esrb_rating = esrb_rating.lower()
        if esrb_rating in ['mature', 'adults only']:
            highest_impact = max(highest_impact, 4)

    # 🔐 Safety fallback: If ESRB is missing and violence tags are present
    if not esrb_rating and highest_impact >= 2:
        highest_impact = max(highest_impact, 3)  # Conservative bump to MA15+

    if highest_impact >= 4:
        return 'R18+'
    elif highest_impact == 3:
        return 'MA15+'
    elif highest_impact == 2:
        return 'M'
    elif highest_impact == 1:
        return 'PG'
    else:
        return 'G'

# --- STEP 4: Get Classification Using RAWG API with Error Handling ---
def get_classification_rating(title):
    try:
        cleaned_title = title.lower().replace(":", "").replace("-", "").strip()

        # --- Step 1: Search RAWG by title ---
        params = {'key': API_KEY, 'search': cleaned_title, 'page_size': 1}
        response = requests.get(BASE_URL, params=params)
        if response.status_code != 200:
            print(f"[ERROR] Search failed for '{title}' — status: {response.status_code}")
            return "Unknown", "Unknown"

        data = response.json()
        results = data.get("results", [])
        if not results or not isinstance(results, list) or not results[0]:
            print(f"[WARN] No search results for '{title}'")
            return "Unknown", "Unknown"

        # Get the slug from the first search result
        game = results[0]
        slug = game.get("slug")
        if not slug:
            print(f"[WARN] No slug in search result for '{title}'")
            return "Unknown", "Unknown"

        # --- Step 2: Fetch details using slug ---
        detail_url = f"{BASE_URL}/{slug}"
        detail_response = requests.get(detail_url, params={'key': API_KEY})
        if detail_response.status_code != 200:
            print(f"[ERROR] Detail fetch failed for '{title}' — status: {detail_response.status_code}")
            return "Unknown", "Unknown"

        try:
            detail_data = detail_response.json()
        except ValueError:
            print(f"[ERROR] Invalid JSON returned for '{title}'")
            return "Unknown", "Unknown"

        # Parse relevant metadata
        tags = [t['name'].lower() for t in detail_data.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in detail_data.get('genres', []) if g.get('name')]
        esrb = detail_data.get("esrb_rating", {}).get("name", "").lower() if detail_data.get("esrb_rating") else ""

        print(f"[INFO] {title} → ESRB: {esrb}, Tags: {tags}, Genres: {genres}")

        acb_rating = classify_acb_rating(tags, genres, esrb_rating=esrb)
        return acb_rating, esrb

    except Exception as e:
        print(f"[EXCEPTION] {title}: {e}")
        return "Unknown", "Unknown"

# --- STEP 5: Apply to First 100 Games ---
acb_ratings = []
violence_levels = []
esrb_labels = []

for idx, title in enumerate(df['game_title'].head(100)):  # Change to full dataset as needed
    print(f"\n[{idx+1}] Processing: {title}")
    acb_rating, esrb = get_classification_rating(title)
    esrb_labels.append(esrb)

    if acb_rating in ['R18+', 'MA15+', 'M']:
        violence = 'Violent'
    elif acb_rating in ['PG', 'G']:
        violence = 'Non-Violent'
    else:
        violence = 'Unknown'

    acb_ratings.append(acb_rating)
    violence_levels.append(violence)
    time.sleep(1)  # Avoid rate limits

# --- STEP 6: Update DataFrame ---
df.loc[df.index[:100], 'acb_rating'] = acb_ratings
df.loc[df.index[:100], 'violence_level_acb'] = violence_levels
df.loc[df.index[:100], 'esrb_rating'] = esrb_labels

# --- STEP 7: Save Output ---
df.to_csv("steam_with_acb_classification.csv", index=False)
print("\n✅ Done! File saved as 'steam_with_acb_classification.csv'")

In [ ]:
import pandas as pd
import requests
import time

# --- STEP 1: Load Your Steam Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

# --- STEP 2: RAWG API Setup ---
API_KEY = "814164033d1945609c5c51925fdb1727"
BASE_URL = "https://api.rawg.io/api/games"

# --- STEP 3: Function to Get Violence Classification ---
def get_violence_label(title):
    try:
        params = {
            'key': API_KEY,
            'search': title,
            'page_size': 1
        }
        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown"

        game = results[0]

        # Extract ESRB
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        # Extract tags and genres
        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]
        combined = tags + genres

        # --- Expanded violence keyword list ---
        violence_keywords = [
            'shooter', 'fps', 'first-person', 'third-person', 'combat', 'battle', 'brawler',
            'military', 'war', 'weapons', 'guns', 'blood', 'gore', 'violence', 'intense violence',
            'tactical', 'pvp', 'pve', 'realistic', 'action', 'killing', 'stealth', 'sniper',
            'assault', 'gunfight', 'mature', 'adults only', 'zombies', 'horror', 'explosions',
            'dismemberment', 'execution', 'psychological horror', 'crime', 'open world', 'gang',
            'crime thriller', 'mob', 'dark', 'criminal', 'fighting', 'deathmatch', 'brutal',
            'narrative violence', 'dungeon', 'melee', 'slaughter', 'weapon-based', 'rage'
        ]

        matched_keywords = [word for word in violence_keywords if word in combined]

        # Final scoring logic
        if esrb_name in ['mature', 'adults only'] or len(matched_keywords) >= 3:
            return "Violent"
        elif len(matched_keywords) == 1 or len(matched_keywords) == 2:
            return "Moderate"
        else:
            return "Non-Violent"

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown"

# --- STEP 4: Apply to First 100 Games ---
violence_labels = []
for idx, title in enumerate(df['game_title'].head(100)):
    print(f"[{idx+1}] Checking: {title}")
    label = get_violence_label(title)
    violence_labels.append(label)
    time.sleep(1)

df.loc[df.index[:100], 'violence_level_api'] = violence_labels

# --- STEP 5: Save to New CSV ---
df.to_csv("steam_with_api_violence_labels.csv", index=False)
print("🎉 Classification complete! File saved as 'steam_with_api_violence_labels.csv'")



###works fine

In [ ]:
import pandas as pd
import requests
import time

# --- STEP 1: Load Your Steam Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

# --- STEP 2: RAWG API Setup ---
API_KEY = "814164033d1945609c5c51925fdb1727"
BASE_URL = "https://api.rawg.io/api/games"

# --- STEP 3: ACB-Style Violence Classification Function ---
def get_classification(title):
    try:
        params = {
            'key': API_KEY,
            'search': title,
            'page_size': 1
        }
        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown", "Unknown"

        game = results[0]

        # Extract ESRB rating
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        # Extract tags and genres
        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]
        combined = tags + genres

        # ACB-style impact keyword mapping
        very_mild = ['cartoon', 'funny', 'family-friendly']
        mild = ['puzzle', 'slapstick', 'platformer', 'fighting']
        moderate = ['shooter', 'fps', 'action', 'first-person', 'battle', 'stealth']
        strong = ['blood', 'military', 'war', 'weapons', 'guns', 'team-based', 'pvp', 'sniper']
        high = ['gore', 'mature', 'torture', 'execution', 'dismemberment', 'graphic violence', 'horror']

        impact_score = 0

        for kw in combined:
            if kw in very_mild:
                impact_score = max(impact_score, 0)
            elif kw in mild:
                impact_score = max(impact_score, 1)
            elif kw in moderate:
                impact_score = max(impact_score, 2)
            elif kw in strong:
                impact_score = max(impact_score, 3)
            elif kw in high:
                impact_score = max(impact_score, 4)

        # Account for ESRB
        if esrb_name in ['mature', 'adults only']:
            impact_score = max(impact_score, 4)

        # ACB rating logic
        if impact_score >= 4:
            acb = 'R18+'
            violence = 'Violent'
        elif impact_score == 3:
            acb = 'MA15+'
            violence = 'Violent'
        elif impact_score == 2:
            acb = 'M'
            violence = 'Moderate'
        elif impact_score == 1:
            acb = 'PG'
            violence = 'Non-Violent'
        else:
            acb = 'G'
            violence = 'Non-Violent'

        return violence, acb

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown", "Unknown"

# --- STEP 4: Classify First 100 Games ---
violence_labels = []
acb_ratings = []

for idx, title in enumerate(df['game_title'].head(100)):
    print(f"[{idx+1}] Checking: {title}")
    violence, acb = get_classification(title)
    violence_labels.append(violence)
    acb_ratings.append(acb)
    time.sleep(1)  # Avoid rate limits

df.loc[df.index[:100], 'violence_level_api'] = violence_labels
df.loc[df.index[:100], 'acb_rating'] = acb_ratings

# --- STEP 5: Save Output ---
df.to_csv("steam_with_acb_ratings.csv", index=False)
print("🎉 File saved as 'steam_with_acb_ratings.csv'")


🔍 Classification Logic
3. Assign Impact Level Based on Keywords
Each tag/genre is checked against 5 lists of keywords (based on ACB guidelines):

Impact	Keywords (Partial list)
Very Mild (score = 0)	cartoon, funny, family-friendly
Mild (score = 1)	puzzle, slapstick, platformer, fighting
Moderate (score = 2)	shooter, fps, battle, stealth
Strong (score = 3)	blood, military, weapons, pvp
High (score = 4)	gore, torture, mature, execution

In [ ]:
import pandas as pd
import requests
import time

# --- STEP 1: Load Your Steam Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

# --- STEP 2: RAWG API Setup ---
API_KEY = "814164033d1945609c5c51925fdb1727"
BASE_URL = "https://api.rawg.io/api/games"

# --- STEP 3: Classification Function with Reason Tags ---
def classify_game(title):
    try:
        params = {
            'key': API_KEY,
            'search': title,
            'page_size': 1
        }
        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown", "Unknown", ""

        game = results[0]
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]
        combined = tags + genres

        # --- Violence Level (flat keywords for parent safety) ---
        violence_keywords = [
            'shooter', 'fps', 'first-person', 'third-person', 'combat', 'battle', 'brawler',
            'military', 'war', 'weapons', 'guns', 'blood', 'gore', 'violence', 'intense violence',
            'tactical', 'pvp', 'pve', 'realistic', 'action', 'killing', 'stealth', 'sniper',
            'assault', 'gunfight', 'mature', 'adults only', 'zombies', 'horror', 'explosions',
            'dismemberment', 'execution', 'psychological horror', 'crime', 'open world', 'gang',
            'crime thriller', 'mob', 'dark', 'criminal', 'fighting', 'deathmatch', 'brutal',
            'narrative violence', 'dungeon', 'melee', 'slaughter', 'weapon-based', 'rage'
        ]

        is_violent = any(word in combined for word in violence_keywords)
        violence_level = 'Violent' if is_violent else 'Non-Violent'

        # --- ACB Rating (impact-based with reason tracking) ---
        very_mild = ['cartoon', 'funny', 'family-friendly']
        mild = ['puzzle', 'slapstick', 'platformer', 'fighting']
        moderate = ['shooter', 'fps', 'action', 'first-person', 'battle', 'stealth']
        strong = ['blood', 'military', 'war', 'weapons', 'guns', 'team-based', 'pvp', 'sniper']
        high = ['gore', 'mature', 'torture', 'execution', 'dismemberment', 'graphic violence', 'horror']

        impact_score = 0
        reason_terms = set()

        for kw in combined:
            if kw in very_mild:
                impact_score = max(impact_score, 0)
            elif kw in mild:
                impact_score = max(impact_score, 1)
                reason_terms.add(kw)
            elif kw in moderate:
                impact_score = max(impact_score, 2)
                reason_terms.add(kw)
            elif kw in strong:
                impact_score = max(impact_score, 3)
                reason_terms.add(kw)
            elif kw in high:
                impact_score = max(impact_score, 4)
                reason_terms.add(kw)

        if esrb_name in ['mature', 'adults only']:
            impact_score = max(impact_score, 4)
            reason_terms.add(esrb_name)

        if impact_score >= 4:
            acb = 'R18+'
        elif impact_score == 3:
            acb = 'MA15+'
        elif impact_score == 2:
            acb = 'M'
        elif impact_score == 1:
            acb = 'PG'
        else:
            acb = 'G'

        return violence_level, acb, ', '.join(sorted(reason_terms))

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown", "Unknown", ""

# --- STEP 4: Apply to First 100 Games ---
violence_levels = []
acb_ratings = []
acb_reasons = []

for idx, title in enumerate(df['game_title'].head(100)):
    print(f"[{idx+1}/100] Checking: {title}")
    violence, acb, reason = classify_game(title)  # ✅ FIXED unpacking
    violence_levels.append(violence)
    acb_ratings.append(acb)
    acb_reasons.append(reason)
    time.sleep(1)  # Respect API rate limit

df.loc[df.index[:100], 'violence_level_api'] = violence_levels
df.loc[df.index[:100], 'acb_rating'] = acb_ratings
df.loc[df.index[:100], 'acb_reason_terms'] = acb_reasons

# --- STEP 5: Save Output ---
df.to_csv("steam_parent_safe_100_with_reasons.csv", index=False)
print("✅ Done! Saved as 'steam_parent_safe_100_with_reasons.csv'")


In [ ]:
import pandas as pd
import requests
import time

# --- Load Full Dataset ---
df = pd.read_csv("steam.csv")
df.rename(columns={'name': 'game_title'}, inplace=True)

API_KEY = "814164033d1945609c5c51925fdb1727"
BASE_URL = "https://api.rawg.io/api/games"

# --- Classification Function ---
def classify_game(title):
    try:
        params = {'key': API_KEY, 'search': title, 'page_size': 1}
        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown", "Unknown", ""

        game = results[0]
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]
        combined = tags + genres

        # --- Violence Detection (flat) ---
        violence_keywords = [
            'shooter', 'fps', 'first-person', 'third-person', 'combat', 'battle', 'brawler',
            'military', 'war', 'weapons', 'guns', 'blood', 'gore', 'violence', 'intense violence',
            'tactical', 'pvp', 'pve', 'realistic', 'action', 'killing', 'stealth', 'sniper',
            'assault', 'gunfight', 'mature', 'adults only', 'zombies', 'horror', 'explosions',
            'dismemberment', 'execution', 'psychological horror', 'crime', 'open world', 'gang',
            'crime thriller', 'mob', 'dark', 'criminal', 'fighting', 'deathmatch', 'brutal',
            'narrative violence', 'dungeon', 'melee', 'slaughter', 'weapon-based', 'rage'
        ]
        is_violent = any(word in combined for word in violence_keywords)
        violence_level = 'Violent' if is_violent else 'Non-Violent'

        # --- ACB Impact Logic ---
        very_mild = ['cartoon', 'funny', 'family-friendly']
        mild = ['puzzle', 'slapstick', 'platformer', 'fighting']
        moderate = ['shooter', 'fps', 'action', 'first-person', 'battle', 'stealth']
        strong = ['blood', 'military', 'war', 'weapons', 'guns', 'team-based', 'pvp', 'sniper']
        high = ['gore', 'mature', 'torture', 'execution', 'dismemberment', 'graphic violence', 'horror']

        impact_score = 0
        reason_terms = set()

        for kw in combined:
            if kw in very_mild:
                impact_score = max(impact_score, 0)
            elif kw in mild:
                impact_score = max(impact_score, 1)
                reason_terms.add(kw)
            elif kw in moderate:
                impact_score = max(impact_score, 2)
                reason_terms.add(kw)
            elif kw in strong:
                impact_score = max(impact_score, 3)
                reason_terms.add(kw)
            elif kw in high:
                impact_score = max(impact_score, 4)
                reason_terms.add(kw)

        if esrb_name in ['mature', 'adults only']:
            impact_score = max(impact_score, 4)
            reason_terms.add(esrb_name)

        if impact_score >= 4:
            acb = 'R18+'
        elif impact_score == 3:
            acb = 'MA15+'
        elif impact_score == 2:
            acb = 'M'
        elif impact_score == 1:
            acb = 'PG'
        else:
            acb = 'G'

        return violence_level, acb, ', '.join(sorted(reason_terms))

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown", "Unknown", ""

# --- Classification Loop ---
violence_levels = []
acb_ratings = []
acb_reasons = []

for idx, title in enumerate(df['game_title']):
    print(f"[{idx+1}/{len(df)}] Checking: {title}")
    violence, acb, reason = classify_game(title)
    violence_levels.append(violence)
    acb_ratings.append(acb)
    acb_reasons.append(reason)
    time.sleep(1)  # Respect API limit

    # Autosave every 500 games
    if (idx + 1) % 500 == 0:
        df.loc[df.index[:idx+1], 'violence_level_api'] = violence_levels
        df.loc[df.index[:idx+1], 'acb_rating'] = acb_ratings
        df.loc[df.index[:idx+1], 'acb_reason_terms'] = acb_reasons
        df.to_csv(f"steam_partial_classified_up_to_{idx+1}.csv", index=False)
        print(f"📝 Autosaved progress at row {idx+1}")

# --- Final Save ---
df['violence_level_api'] = violence_levels
df['acb_rating'] = acb_ratings
df['acb_reason_terms'] = acb_reasons
df.to_csv("steam_full_with_acb_classification.csv", index=False)
print("✅ Full classification complete! Saved as 'steam_full_with_acb_classification.csv'")


In [ ]:
import pandas as pd

# Load the full Steam dataset
df = pd.read_csv("steam.csv")

# Extract rows from 15,857 to the end (row 27,075)
df_split = df.iloc[15855:].copy()

# Save to a new CSV file
df_split.to_csv("steam_rows_15856_to_27075.csv", index=False)

print("✅ File saved as 'steam_rows_15857_to_27075.csv'")


In [9]:
import pandas as pd
import requests
import time

# --- Load the Split Dataset ---
df = pd.read_csv("steam_rows_15856_to_27075.csv")
df.rename(columns={'name': 'game_title'}, inplace=True, errors='ignore')

# --- RAWG API Setup ---
API_KEY = "bc68ec531e7d4da389c39b49a82d6d3f"
BASE_URL = "https://api.rawg.io/api/games"

# --- Classification Function ---
def classify_game(title):
    try:
        params = {'key': API_KEY, 'search': title, 'page_size': 1}
        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print(f"API error for '{title}': Status {response.status_code}")
            return "Unknown", "Unknown", ""

        data = response.json()
        results = data.get("results")
        if not results or results[0] is None:
            return "Unknown", "Unknown", ""

        game = results[0]
        esrb = game.get("esrb_rating", {})
        esrb_name = esrb.get("name", "").lower() if esrb else ""

        tags = [t['name'].lower() for t in game.get('tags', []) if t.get('name')]
        genres = [g['name'].lower() for g in game.get('genres', []) if g.get('name')]
        combined = tags + genres

        # --- Violence Detection ---
        violence_keywords = [
            'shooter', 'fps', 'first-person', 'third-person', 'combat', 'battle', 'brawler',
            'military', 'war', 'weapons', 'guns', 'blood', 'gore', 'violence', 'intense violence',
            'tactical', 'pvp', 'pve', 'realistic', 'action', 'killing', 'stealth', 'sniper',
            'assault', 'gunfight', 'mature', 'adults only', 'zombies', 'horror', 'explosions',
            'dismemberment', 'execution', 'psychological horror', 'crime', 'open world', 'gang',
            'crime thriller', 'mob', 'dark', 'criminal', 'fighting', 'deathmatch', 'brutal',
            'narrative violence', 'dungeon', 'melee', 'slaughter', 'weapon-based', 'rage'
        ]
        is_violent = any(word in combined for word in violence_keywords)
        violence_level = 'Violent' if is_violent else 'Non-Violent'

        # --- ACB Impact Logic ---
        very_mild = ['cartoon', 'funny', 'family-friendly']
        mild = ['puzzle', 'slapstick', 'platformer', 'fighting']
        moderate = ['shooter', 'fps', 'action', 'first-person', 'battle', 'stealth']
        strong = ['blood', 'military', 'war', 'weapons', 'guns', 'team-based', 'pvp', 'sniper']
        high = ['gore', 'mature', 'torture', 'execution', 'dismemberment', 'graphic violence', 'horror']

        impact_score = 0
        reason_terms = set()

        for kw in combined:
            if kw in very_mild:
                impact_score = max(impact_score, 0)
            elif kw in mild:
                impact_score = max(impact_score, 1)
                reason_terms.add(kw)
            elif kw in moderate:
                impact_score = max(impact_score, 2)
                reason_terms.add(kw)
            elif kw in strong:
                impact_score = max(impact_score, 3)
                reason_terms.add(kw)
            elif kw in high:
                impact_score = max(impact_score, 4)
                reason_terms.add(kw)

        if esrb_name in ['mature', 'adults only']:
            impact_score = max(impact_score, 4)
            reason_terms.add(esrb_name)

        if impact_score >= 4:
            acb = 'R18+'
        elif impact_score == 3:
            acb = 'MA15+'
        elif impact_score == 2:
            acb = 'M'
        elif impact_score == 1:
            acb = 'PG'
        else:
            acb = 'G'

        return violence_level, acb, ', '.join(sorted(reason_terms))

    except Exception as e:
        print(f"Error with title '{title}': {e}")
        return "Unknown", "Unknown", ""

# --- Classification Loop ---
violence_levels = []
acb_ratings = []
acb_reasons = []

for idx, title in enumerate(df['game_title']):
    print(f"[{idx+1}/{len(df)}] Checking: {title}")
    violence, acb, reason = classify_game(title)
    violence_levels.append(violence)
    acb_ratings.append(acb)
    acb_reasons.append(reason)
    time.sleep(1)  # API rate-limit safe

    # Optional: Autosave every 500 titles
    if (idx + 1) % 500 == 0:
        df.loc[df.index[:idx+1], 'violence_level_api'] = violence_levels
        df.loc[df.index[:idx+1], 'acb_rating'] = acb_ratings
        df.loc[df.index[:idx+1], 'acb_reason_terms'] = acb_reasons
        df.to_csv(f"steam_partial_classified_up_to_{15856 + idx + 1}.csv", index=False)
        print(f"📝 Autosaved progress at row {15856 + idx + 1}")

# --- Final Save ---
df['violence_level_api'] = violence_levels
df['acb_rating'] = acb_ratings
df['acb_reason_terms'] = acb_reasons
df.to_csv("steam_partial_classified_up_to_27075.csv", index=False)
print("✅ Done! Saved as 'steam_partial_classified_up_to_27075.csv'")


[1/11220] Checking: The Bounty: Deluxe Edition
[2/11220] Checking: Galaxy of Drones
[3/11220] Checking: Benjamin Johnson EP.1
[4/11220] Checking: Swords and Sandals Medieval
[5/11220] Checking: High Hell
[6/11220] Checking: WILOO
[7/11220] Checking: Acorns Above: A World Gone Nuts
[8/11220] Checking: StarBallMadNess
[9/11220] Checking: Floor Plan: Hands-On Edition
[10/11220] Checking: The Ranger: Lost Tribe
[11/11220] Checking: AMID EVIL
[12/11220] Checking: Infinite Skyline
[13/11220] Checking: All-Star Fruit Racing
[14/11220] Checking: Frontier Pilot Simulator
[15/11220] Checking: Joumee The Hedgehog
[16/11220] Checking: Galaxy Annihilation
[17/11220] Checking: Exterminator: Escape!
[18/11220] Checking: Starblast
[19/11220] Checking: GyroSphere Trials
[20/11220] Checking: Alien Splatter Redux
[21/11220] Checking: Adam Waste
[22/11220] Checking: Black Mist
[23/11220] Checking: Mahjong Magic Islands
[24/11220] Checking: TimeLock VR
[25/11220] Checking: Timbertales
[26/11220] Checking: 